# Phase 03I: Teacher C SigLIP ViT-B/16 Offline Preflight

Mục tiêu của notebook là khóa nguồn pretrained ngoại tuyến, preprocessing,
kiến trúc Teacher C và khả năng tương thích với shared decoder của Phase 2A.
Notebook **không huấn luyện**, **không tạo optimizer** và **không mở Held-out
Test**. Chạy tuần tự từ Cell 1 đến Cell 11.


## Cell 1 — Môi trường và khám phá input

**Công việc:** bật strict deterministic; tìm Phase 1, Phase 2A và Phase 3A.

**Đầu ra dự kiến:** `PASSED`, GPU được nhận, Test vẫn `SEALED`, tiến độ `1/11`.


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = (
    KAGGLE_WORKING_ROOT
    / ""
    / "phase3_teacher_c_siglip_preflight"
)
CONFIG_DIR = OUTPUT_ROOT / "configs"
AUDIT_DIR = OUTPUT_ROOT / "audits"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    CONFIG_DIR,
    AUDIT_DIR,
    CHECKPOINT_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with open(path, "r", encoding="utf-8") as source:
        return json.load(source)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        while True:
            chunk = source.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    candidate = (root / relative_path).resolve()
    assert candidate.is_relative_to(root), (
        f"Path escapes root: {relative_path}"
    )
    return candidate


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)
    if torch.cuda.is_available():
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_math_sdp(True)


assert KAGGLE_INPUT_ROOT.is_dir(), (
    "Cell này phải chạy trên Kaggle."
)
assert torch.cuda.is_available(), (
    "Hãy bật GPU trong Settings → Accelerator."
)
set_deterministic(SEED)
device = torch.device("cuda")

phase1_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE1_FORMAL_DELIVERABLES_READY.json"
    )
)
phase2a_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
    )
)
phase3a_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE3_TEACHER_POOL_INPUT_READY.json"
    )
)
assert len(phase1_matches) == 1, (
    f"Cần đúng 1 Phase 1 marker, thấy {len(phase1_matches)}."
)
assert len(phase2a_matches) == 1, (
    f"Cần đúng 1 Phase 2A marker, thấy {len(phase2a_matches)}."
)
assert len(phase3a_matches) == 1, (
    f"Cần đúng 1 Phase 3A marker, thấy {len(phase3a_matches)}."
)

PHASE1_READY_PATH = phase1_matches[0].resolve()
PHASE2A_READY_PATH = phase2a_matches[0].resolve()
PHASE3A_READY_PATH = phase3a_matches[0].resolve()
PHASE1_ROOT = PHASE1_READY_PATH.parent
PHASE2A_ROOT = PHASE2A_READY_PATH.parent
PHASE3A_ROOT = PHASE3A_READY_PATH.parent

audit_path = AUDIT_DIR / "phase3i_cell01_environment_discovery.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "environment": {
            "python": platform.python_version(),
            "pytorch": torch.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
            "seed": SEED,
            "strict_deterministic": True,
        },
        "inputs": {
            "phase1_marker": str(PHASE1_READY_PATH),
            "phase2a_marker": str(PHASE2A_READY_PATH),
            "phase3a_marker": str(PHASE3A_READY_PATH),
        },
        "data_policy": {
            "parquet_opened": False,
            "image_read": False,
            "test_set_status": "sealed",
            "test_set_used": False,
            "flickr_used": False,
            "training_started": False,
        },
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3I CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — SigLIP ViT-B/16")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Phase 1 marker            : FOUND")
print("Phase 2A marker           : FOUND")
print("Phase 3A marker           : FOUND")
print("Parquet opened            : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Training started          : False")
print("Audit                     :", audit_path)
print("Progress                  : 1 / 11 cells passed")
print("Next                      : verify Phase 3A transition bundle")


PHASE 3I CELL 1 STATUS
Status                    : PASSED
Teacher                   : A — SigLIP ViT-B/16
GPU                       : Tesla T4
Phase 1 marker            : FOUND
Phase 2A marker           : FOUND
Phase 3A marker           : FOUND
Parquet opened            : False
Test Set status           : SEALED
Test Set used             : False
Training started          : False
Audit                     : /kaggle/working//phase3_teacher_c_siglip_preflight/audits/phase3i_cell01_environment_discovery.json
Progress                  : 1 / 11 cells passed
Next                      : verify Phase 3A transition bundle


## Cell 2 — Xác minh Phase 03H và quyền vào Teacher C

**Công việc:** kiểm checksum bundle Phase 03H, marker hoàn tất Teacher B và
contract cho phép chạy Teacher C.

**Đầu ra dự kiến:** 13/13 artifact hợp lệ; Teacher C khớp S3 TinyViT-5M;
repo gợi ý là `google/siglip-base-patch16-224`.


In [2]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "KAGGLE_INPUT_ROOT", "PHASE3A_READY_PATH", "PHASE3A_ROOT",
    "AUDIT_DIR", "read_json", "write_json", "sha256_file",
    "resolve_inside",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1: {missing}"

phase3h_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE3_TEACHER_B_THREE_SEED_AGGREGATE_COMPLETE.json"
    )
)
authorization_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE3_TEACHER_C_PREFLIGHT_AUTHORIZED.json"
    )
)
assert len(phase3h_matches) == 1, (
    f"Cần đúng 1 Phase 03H completion marker, thấy {len(phase3h_matches)}."
)
assert len(authorization_matches) == 1, (
    f"Cần đúng 1 Teacher C authorization marker, thấy "
    f"{len(authorization_matches)}."
)
PHASE3H_COMPLETE_PATH = phase3h_matches[0].resolve()
PHASE3H_ROOT = PHASE3H_COMPLETE_PATH.parent
TEACHER_C_AUTHORIZATION_PATH = authorization_matches[0].resolve()
assert TEACHER_C_AUTHORIZATION_PATH.parent == PHASE3H_ROOT

completion = read_json(PHASE3H_COMPLETE_PATH)
authorization = read_json(TEACHER_C_AUTHORIZATION_PATH)
assert completion["status"] == "complete"
assert completion["teacher_id"] == "B"
assert completion["run_count"] == 3
assert completion["all_quality_gates_passed"] is True
assert completion["all_qualified_for_kd"] is True
assert completion["test_set_status"] == "sealed"
assert completion["test_set_used"] is False
assert authorization["status"] == "authorized"
assert authorization["next_phase"] == "Phase 03I Teacher C offline preflight"
assert authorization["training_started"] is False
assert authorization["test_set_status"] == "sealed"

manifest_record = completion["manifest"]["csv"]
manifest_path = resolve_inside(
    PHASE3H_ROOT,
    manifest_record["relative_path"],
)
assert manifest_path.name == "phase3h_manifest.csv"
assert sha256_file(manifest_path) == manifest_record["sha256"]
with open(manifest_path, "r", encoding="utf-8", newline="") as source:
    phase3h_manifest_rows = list(csv.DictReader(source))
assert len(phase3h_manifest_rows) == 13
for row in phase3h_manifest_rows:
    artifact = resolve_inside(PHASE3H_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

TEACHER_C_ENTRY_CONTRACT_PATH = (
    PHASE3H_ROOT / "configs" / "teacher_c_entry_contract.json"
)
assert sha256_file(TEACHER_C_ENTRY_CONTRACT_PATH) == (
    authorization["entry_contract"]["sha256"]
)
TEACHER_C_ENTRY_CONTRACT = read_json(TEACHER_C_ENTRY_CONTRACT_PATH)
assert TEACHER_C_ENTRY_CONTRACT["status"] == "authorized_for_preflight"
assert TEACHER_C_ENTRY_CONTRACT["teacher_id"] == "C"
assert TEACHER_C_ENTRY_CONTRACT["matched_student"]["student_id"] == "S3"
assert (
    TEACHER_C_ENTRY_CONTRACT["candidate"]["suggested_repo_id"]
    == "google/siglip-base-patch16-224"
)

phase3_ready = read_json(PHASE3A_READY_PATH)
assert phase3_ready["status"] == "ready"
assert phase3_ready["test_set_status"] == "sealed"
assert phase3_ready["test_set_used"] is False
teacher_c_entry = phase3_ready["teacher_families"]["C"]
assert teacher_c_entry["matched_student_id"] == "S3"
ENTRY_CONTRACT_PATH = (
    PHASE3A_ROOT / "configs" /
    "phase3_teacher_pool_entry_contract.json"
)
ENTRY_CONTRACT = read_json(ENTRY_CONTRACT_PATH)
assert ENTRY_CONTRACT["status"] == "locked"
assert ENTRY_CONTRACT["dataset"]["test_set_status"] == "sealed"

audit_path = AUDIT_DIR / "phase3i_cell02_phase3h_authorization.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase3h_manifest_rows_verified": len(phase3h_manifest_rows),
        "teacher_b_run_count": 3,
        "teacher_b_all_qualified_for_kd": True,
        "teacher_c_entry_contract": str(TEACHER_C_ENTRY_CONTRACT_PATH),
        "teacher_c": teacher_c_entry,
        "suggested_repo_id": "google/siglip-base-patch16-224",
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 03I CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 03H                 : VERIFIED")
print("Phase 03H manifest        : 13 / 13 VERIFIED")
print("Teacher B runs            : 3 / 3 KD-QUALIFIED")
print("Teacher C authorization   : VERIFIED")
print("Teacher C matched student : S3 — TinyViT-5M")
print("Suggested repository      : google/siglip-base-patch16-224")
print("Test Set status           : SEALED")
print("Audit                     :", audit_path)
print("Progress                  : 2 / 11 cells passed")
print("Next                      : verify Phase 1 and Phase 2A artifacts")


PHASE 03I CELL 2 STATUS
Status                    : PASSED
Phase 03H                 : VERIFIED
Phase 03H manifest        : 13 / 13 VERIFIED
Teacher B runs            : 3 / 3 KD-QUALIFIED
Teacher C authorization   : VERIFIED
Teacher C matched student : S3 — TinyViT-5M
Suggested repository      : google/siglip-base-patch16-224
Test Set status           : SEALED
Audit                     : /kaggle/working//phase3_teacher_c_siglip_preflight/audits/phase3i_cell02_phase3h_authorization.json
Progress                  : 2 / 11 cells passed
Next                      : verify Phase 1 and Phase 2A artifacts


## Cell 3 — Khóa Phase 1 và Phase 2A artifacts

**Công việc:** xác minh 32 checksum Phase 1, manifest Phase 2A, vocabulary,
sequence contract và shared decoder checkpoint.

**Đầu ra dự kiến:** toàn bộ checksum `PASSED`; Test Parquet chỉ xác nhận tồn
tại, không được mở.


In [3]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "PHASE1_ROOT",
    "PHASE2A_ROOT",
    "PHASE2A_READY_PATH",
    "AUDIT_DIR",
    "read_json",
    "write_json",
    "sha256_file",
    "resolve_inside",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–2: {missing}"


def exactly_one(root, pattern):
    matches = sorted(Path(root).rglob(pattern))
    assert len(matches) == 1, (
        f"Cần đúng một {pattern}, thấy {len(matches)}: {matches[:5]}"
    )
    return matches[0].resolve()


PHASE1_VALIDATION_PARQUET = exactly_one(
    PHASE1_ROOT, "mscoco_validation.parquet"
)
PHASE1_IMAGE_MANIFEST_PARQUET = exactly_one(
    PHASE1_ROOT, "mscoco_image_manifest.parquet"
)
PHASE1_TEST_PARQUET = exactly_one(
    PHASE1_ROOT, "mscoco_test.parquet"
)
PHASE1_VOCAB_PATH = exactly_one(
    PHASE1_ROOT, "word_vocabulary_minfreq5.json"
)
PHASE1_TOKENIZER_PATH = exactly_one(
    PHASE1_ROOT, "tokenizer_contract.json"
)
PHASE1_SEQUENCE_PATH = exactly_one(
    PHASE1_ROOT, "sequence_encoding_contract.json"
)
PHASE1_SHA256SUMS_PATH = exactly_one(
    PHASE1_ROOT, "SHA256SUMS.txt"
)

checksum_rows = {}
for raw_line in PHASE1_SHA256SUMS_PATH.read_text(
    encoding="utf-8"
).splitlines():
    if not raw_line.strip():
        continue
    checksum, relative_path = raw_line.split("  ", 1)
    checksum_rows[relative_path.replace("\\", "/")] = checksum
assert len(checksum_rows) == 32
for relative_path, checksum in checksum_rows.items():
    artifact = resolve_inside(PHASE1_ROOT, relative_path)
    assert artifact.is_file()
    assert sha256_file(artifact) == checksum

vocab_payload = read_json(PHASE1_VOCAB_PATH)
sequence_payload = read_json(PHASE1_SEQUENCE_PATH)
vocab_size = int(
    vocab_payload.get(
        "vocabulary_size",
        len(
            vocab_payload.get(
                "id_to_token",
                vocab_payload.get("tokens", []),
            )
        ),
    )
)
assert vocab_size == 10080
maximum_decoder_steps = int(
    sequence_payload.get(
        "maximum_decoder_steps",
        sequence_payload.get("length_policy", {}).get(
            "maximum_decoder_time_steps", 31
        ),
    )
)
assert maximum_decoder_steps == 31

phase2a_ready = read_json(PHASE2A_READY_PATH)
assert phase2a_ready["status"] == "ready"
assert phase2a_ready["relocatable"] is True
assert phase2a_ready["data_policy"]["test_set_status"] == "sealed"
assert phase2a_ready["data_policy"]["test_set_used"] is False

manifest_record = phase2a_ready["manifests"]["csv"]
phase2a_manifest_path = resolve_inside(
    PHASE2A_ROOT,
    manifest_record["relative_path"],
)
assert sha256_file(phase2a_manifest_path) == manifest_record["sha256"]
with open(
    phase2a_manifest_path,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    phase2a_rows = list(csv.DictReader(source))
assert len(phase2a_rows) == int(phase2a_ready["artifact_count"])
for row in phase2a_rows:
    artifact = resolve_inside(
        PHASE2A_ROOT,
        row["relative_path"],
    )
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

required_artifacts = phase2a_ready["required_artifacts"]
SHARED_DECODER_CHECKPOINT_PATH = resolve_inside(
    PHASE2A_ROOT,
    required_artifacts["shared_decoder_checkpoint"],
)
SHARED_DECODER_CONTRACT_PATH = exactly_one(
    PHASE2A_ROOT,
    "shared_transformer_decoder_contract.json",
)
assert SHARED_DECODER_CHECKPOINT_PATH.is_file()
assert SHARED_DECODER_CONTRACT_PATH.is_file()

audit_path = AUDIT_DIR / "phase3i_cell03_upstream_artifacts.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase1_checksum_rows": len(checksum_rows),
        "phase2a_manifest_rows": len(phase2a_rows),
        "vocabulary_size": vocab_size,
        "maximum_decoder_steps": maximum_decoder_steps,
        "shared_decoder_checkpoint": {
            "path": str(SHARED_DECODER_CHECKPOINT_PATH),
            "sha256": sha256_file(
                SHARED_DECODER_CHECKPOINT_PATH
            ),
        },
        "test_parquet_exists": PHASE1_TEST_PARQUET.is_file(),
        "test_parquet_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3I CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 1 checksums         : 32 / 32 PASSED")
print("Phase 2A manifest         :", len(phase2a_rows), "/", len(phase2a_rows), "PASSED")
print("Vocabulary size           :", vocab_size)
print("Maximum decoder steps     :", maximum_decoder_steps)
print("Shared decoder checkpoint : VERIFIED")
print("Test Parquet opened       : False")
print("Test Set status           : SEALED")
print("Audit                     :", audit_path)
print("Progress                  : 3 / 11 cells passed")
print("Next                      : discover and lock offline SigLIP weights")


PHASE 3I CELL 3 STATUS
Status                    : PASSED
Phase 1 checksums         : 32 / 32 PASSED
Phase 2A manifest         : 38 / 38 PASSED
Vocabulary size           : 10080
Maximum decoder steps     : 31
Shared decoder checkpoint : VERIFIED
Test Parquet opened       : False
Test Set status           : SEALED
Audit                     : /kaggle/working//phase3_teacher_c_siglip_preflight/audits/phase3i_cell03_upstream_artifacts.json
Progress                  : 3 / 11 cells passed
Next                      : discover and lock offline SigLIP weights


## Cell 4 — Khóa nguồn SigLIP pretrained ngoại tuyến

**Công việc:** tìm đúng một package SigLIP, xác minh
`MODEL_SOURCE_MANIFEST.json`, immutable commit, config và weights.

**Đầu ra dự kiến:** hidden 768, patch 16, image 224, 12 layer; không tải mạng.


In [4]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "KAGGLE_INPUT_ROOT", "CONFIG_DIR", "AUDIT_DIR",
    "read_json", "write_json", "sha256_file", "resolve_inside",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–3: {missing}"

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
from transformers import SiglipConfig

candidate_manifests = []
for manifest_path in KAGGLE_INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        manifest = read_json(manifest_path)
    except Exception:
        continue
    if manifest.get("repo_id") == "google/siglip-base-patch16-224":
        candidate_manifests.append((manifest_path.resolve(), manifest))
assert len(candidate_manifests) == 1, (
    "Cần gắn đúng một offline package cho "
    "google/siglip-base-patch16-224. "
    f"Tìm thấy {len(candidate_manifests)}: "
    f"{[str(item[0]) for item in candidate_manifests]}"
)
SIGLIP_SOURCE_MANIFEST_PATH, SIGLIP_SOURCE_MANIFEST = (
    candidate_manifests[0]
)
SIGLIP_SOURCE_DIR = SIGLIP_SOURCE_MANIFEST_PATH.parent
assert SIGLIP_SOURCE_MANIFEST["schema_version"] == (
    "offline_huggingface_model_source_v1"
)
assert SIGLIP_SOURCE_MANIFEST["repo_id"] == (
    "google/siglip-base-patch16-224"
)
resolved_commit = str(SIGLIP_SOURCE_MANIFEST["resolved_commit"])
assert len(resolved_commit) == 40
assert all(character in "0123456789abcdef" for character in resolved_commit)

source_rows = SIGLIP_SOURCE_MANIFEST["files"]
assert source_rows
for row in source_rows:
    artifact = resolve_inside(SIGLIP_SOURCE_DIR, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

config_path = SIGLIP_SOURCE_DIR / "config.json"
processor_path = SIGLIP_SOURCE_DIR / "preprocessor_config.json"
assert config_path.is_file()
assert processor_path.is_file()
SIGLIP_CONFIG_RAW = read_json(config_path)
resolved_config = SiglipConfig.from_pretrained(
    SIGLIP_SOURCE_DIR,
    local_files_only=True,
)
vision_config = resolved_config.vision_config
assert int(vision_config.hidden_size) == 768
assert int(vision_config.patch_size) == 16
assert int(vision_config.image_size) == 224
assert int(vision_config.num_hidden_layers) == 12
SIGLIP_WEIGHT_FILES = sorted(
    list(SIGLIP_SOURCE_DIR.glob("*.safetensors"))
    + list(SIGLIP_SOURCE_DIR.glob("pytorch_model*.bin"))
)
assert SIGLIP_WEIGHT_FILES

source_contract_path = (
    CONFIG_DIR / "teacher_c_siglip_pretrained_source_contract.json"
)
write_json(
    source_contract_path,
    {
        "contract_version": "phase3_teacher_c_source_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "model_identifier": "google/siglip-base-patch16-224",
        "resolved_commit": resolved_commit,
        "local_source_dir": str(SIGLIP_SOURCE_DIR),
        "source_manifest_sha256": sha256_file(
            SIGLIP_SOURCE_MANIFEST_PATH
        ),
        "source_file_count": len(source_rows),
        "architecture": {
            "hidden_size": 768,
            "patch_size": 16,
            "image_size": 224,
            "num_hidden_layers": 12,
            "visual_tokens": 196,
        },
        "network_download": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

audit_path = AUDIT_DIR / "phase3i_cell04_siglip_source.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "resolved_commit": resolved_commit,
        "source_manifest_rows_verified": len(source_rows),
        "weight_file_count": len(SIGLIP_WEIGHT_FILES),
        "source_contract": str(source_contract_path),
        "source_contract_sha256": sha256_file(source_contract_path),
        "network_download": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 03I CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Model identifier          : google/siglip-base-patch16-224")
print("Resolved commit           :", resolved_commit)
print("Source manifest           :", SIGLIP_SOURCE_MANIFEST_PATH)
print("Source files              :", len(source_rows), "/", len(source_rows), "VERIFIED")
print("Hidden / patch / image    : 768 / 16 / 224")
print("Visual tokens             : 196")
print("Network download          : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 11 cells passed")
print("Next                      : build Validation-only indexes")


PHASE 03I CELL 4 STATUS
Status                    : PASSED
Model identifier          : google/siglip-base-patch16-224
Resolved commit           : 7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed
Source manifest           : /kaggle/input/models/nguyentamphuc/siglip-base-patch16-224-offline/pytorch/default/1/google_siglip_base_patch16_224_7fd15f06/MODEL_SOURCE_MANIFEST.json
Source files              : 4 / 4 VERIFIED
Hidden / patch / image    : 768 / 16 / 224
Visual tokens             : 196
Network download          : False
Test Set status           : SEALED
Progress                  : 4 / 11 cells passed
Next                      : build Validation-only indexes


## Cell 5 — Validation-only image và token index

**Công việc:** dựng 5.000 ảnh Validation và 25.010 caption token; ánh xạ đúng
đường dẫn COCO; không đọc caption/test payload.

**Đầu ra dự kiến:** 5.000 ảnh, 25.010 caption, overlap với Test bằng 0.


In [5]:
CELL_STARTED_AT = time.perf_counter()
import pyarrow.parquet as pq

required = [
    "PHASE1_VALIDATION_PARQUET",
    "PHASE1_IMAGE_MANIFEST_PARQUET",
    "KAGGLE_INPUT_ROOT",
    "AUDIT_DIR",
    "write_json",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–4: {missing}"

validation_schema = pq.ParquetFile(
    PHASE1_VALIDATION_PARQUET
).schema_arrow
manifest_schema = pq.ParquetFile(
    PHASE1_IMAGE_MANIFEST_PARQUET
).schema_arrow
validation_columns = set(validation_schema.names)
manifest_columns = set(manifest_schema.names)
assert {"image_id", "token_ids"}.issubset(validation_columns)

validation_table = pq.read_table(
    PHASE1_VALIDATION_PARQUET,
    columns=["image_id", "token_ids"],
)
manifest_required = {
    "image_id",
    "experiment_split",
    "source_partition",
    "image_path",
}
assert manifest_required.issubset(manifest_columns)
manifest_table = pq.read_table(
    PHASE1_IMAGE_MANIFEST_PARQUET,
    columns=sorted(manifest_required),
)
manifest_rows = manifest_table.to_pylist()
validation_manifest_rows = [
    row for row in manifest_rows
    if row["experiment_split"] == "validation"
]
test_ids = {
    int(row["image_id"])
    for row in manifest_rows
    if row["experiment_split"] == "held_out_test"
}
assert len(validation_manifest_rows) == 5000
assert len(test_ids) == 5000

validation_image_ids = [
    int(value)
    for value in validation_table["image_id"].to_pylist()
]
validation_token_ids = (
    validation_table["token_ids"].to_pylist()
)
assert len(validation_image_ids) == 25010
assert len(set(validation_image_ids)) == 5000
assert not (set(validation_image_ids) & test_ids)

val_leaf_candidates = []
for candidate in KAGGLE_INPUT_ROOT.rglob("val2014"):
    if not candidate.is_dir():
        continue
    probe = next(
        candidate.glob("COCO_val2014_*.jpg"),
        None,
    )
    if probe is not None:
        val_leaf_candidates.append(candidate.resolve())
assert len(val_leaf_candidates) == 1, (
    f"Cần đúng một val2014 image leaf, thấy {val_leaf_candidates}"
)
COCO_VAL_LEAF = val_leaf_candidates[0]

VALIDATION_PATH_BY_ID = {}
for row in validation_manifest_rows:
    image_id = int(row["image_id"])
    assert row["source_partition"] == "val2014"
    file_name = Path(row["image_path"]).name
    image_path = COCO_VAL_LEAF / file_name
    assert image_path.is_file(), f"Thiếu ảnh: {image_path}"
    VALIDATION_PATH_BY_ID[image_id] = image_path
assert len(VALIDATION_PATH_BY_ID) == 5000

VALIDATION_RECORDS = [
    {
        "image_id": int(image_id),
        "token_ids": [int(token) for token in tokens],
    }
    for image_id, tokens in zip(
        validation_image_ids,
        validation_token_ids,
    )
]

audit_path = AUDIT_DIR / "phase3i_cell05_validation_index.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_caption_rows": len(VALIDATION_RECORDS),
        "validation_unique_images": len(VALIDATION_PATH_BY_ID),
        "held_out_test_ids": len(test_ids),
        "validation_test_overlap": 0,
        "caption_text_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3I CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation captions       :", len(VALIDATION_RECORDS))
print("Validation images         :", len(VALIDATION_PATH_BY_ID))
print("Held-out Test IDs         :", len(test_ids))
print("Held-out Test overlap     : 0")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test Set status           : SEALED")
print("Audit                     :", audit_path)
print("Progress                  : 5 / 11 cells passed")
print("Next                      : validate SigLIP preprocessing")


PHASE 3I CELL 5 STATUS
Status                    : PASSED
Validation captions       : 25010
Validation images         : 5000
Held-out Test IDs         : 5000
Held-out Test overlap     : 0
Caption text read         : False
Test Parquet opened       : False
Test Set status           : SEALED
Audit                     : /kaggle/working//phase3_teacher_c_siglip_preflight/audits/phase3i_cell05_validation_index.json
Progress                  : 5 / 11 cells passed
Next                      : validate SigLIP preprocessing


## Cell 6 — SigLIP preprocessing contract

**Công việc:** kiểm RGB 224×224, resize bicubic, mean/std 0.5 và tính lặp lại.

**Đầu ra dự kiến:** tensor `[3,3,224,224]`, finite, repeat error bằng 0.


In [6]:
CELL_STARTED_AT = time.perf_counter()
from PIL import Image
from transformers import SiglipImageProcessor

required = [
    "SIGLIP_SOURCE_DIR", "VALIDATION_PATH_BY_ID", "CONFIG_DIR",
    "AUDIT_DIR", "write_json", "sha256_file",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–5: {missing}"

SIGLIP_PROCESSOR = SiglipImageProcessor.from_pretrained(
    SIGLIP_SOURCE_DIR,
    local_files_only=True,
)
assert SIGLIP_PROCESSOR.size == {"height": 224, "width": 224}
expected_mean = [0.5, 0.5, 0.5]
expected_std = [0.5, 0.5, 0.5]
assert np.allclose(SIGLIP_PROCESSOR.image_mean, expected_mean, atol=1e-8)
assert np.allclose(SIGLIP_PROCESSOR.image_std, expected_std, atol=1e-8)

sample_ids = sorted(VALIDATION_PATH_BY_ID)[:3]
sample_images = []
for image_id in sample_ids:
    with Image.open(VALIDATION_PATH_BY_ID[image_id]) as image:
        sample_images.append(image.convert("RGB").copy())
first = SIGLIP_PROCESSOR(
    images=sample_images, return_tensors="pt"
)["pixel_values"]
second = SIGLIP_PROCESSOR(
    images=sample_images, return_tensors="pt"
)["pixel_values"]
repeat_error = (first - second).abs().max().item()
assert list(first.shape) == [3, 3, 224, 224]
assert torch.isfinite(first).all()
assert repeat_error == 0.0

contract_path = (
    CONFIG_DIR / "teacher_c_siglip_preprocessing_contract.json"
)
write_json(
    contract_path,
    {
        "contract_version": "phase3_teacher_c_preprocessing_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "processor_class": "SiglipImageProcessor",
        "input_mode": "RGB",
        "size": [224, 224],
        "resample": "bicubic",
        "mean": expected_mean,
        "std": expected_std,
        "preflight_augmentation": False,
        "network_download": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

audit_path = AUDIT_DIR / "phase3i_cell06_preprocessing.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "sample_image_ids": sample_ids,
        "output_shape": list(first.shape),
        "repeat_maximum_error": repeat_error,
        "output_finite": True,
        "contract": str(contract_path),
        "contract_sha256": sha256_file(contract_path),
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 03I CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Processor                 : SiglipImageProcessor")
print("Pixel tensor              :", list(first.shape))
print("SigLIP mean / std         :", expected_mean, "/", expected_std)
print("Repeat maximum error      :", repeat_error)
print("Network download          : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 11 cells passed")
print("Next                      : reconstruct Teacher C architecture")


PHASE 03I CELL 6 STATUS
Status                    : PASSED
Processor                 : SiglipImageProcessor
Pixel tensor              : [3, 3, 224, 224]
SigLIP mean / std         : [0.5, 0.5, 0.5] / [0.5, 0.5, 0.5]
Repeat maximum error      : 0.0
Network download          : False
Test Set status           : SEALED
Progress                  : 6 / 11 cells passed
Next                      : reconstruct Teacher C architecture


## Cell 7 — Dựng Teacher C và strict-load shared decoder

**Công việc:** dựng SigLIP vision encoder, adapter 768→256 cho 196 visual
tokens, rồi strict-load shared Transformer decoder.

**Đầu ra dự kiến:** memory `[2,196,256]`, logits `[2,T,10080]`, thiếu/thừa key
bằng 0.


In [7]:
CELL_STARTED_AT = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from transformers import SiglipModel

required = [
    "SIGLIP_SOURCE_DIR",
    "SHARED_DECODER_CHECKPOINT_PATH",
    "SIGLIP_PROCESSOR",
    "VALIDATION_RECORDS",
    "VALIDATION_PATH_BY_ID",
    "device",
    "SEED",
    "CONFIG_DIR",
    "AUDIT_DIR",
    "CHECKPOINT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–6: {missing}"

DECODER_CONFIG = {
    "vocabulary_size": 10080,
    "pad_id": 0,
    "bos_id": 1,
    "eos_id": 2,
    "unk_id": 3,
    "d_model": 256,
    "nhead": 8,
    "head_dimension": 32,
    "num_layers": 3,
    "dim_feedforward": 1024,
    "dropout": 0.1,
    "activation": "gelu",
    "maximum_decoder_steps": 31,
    "memory_dimension": 256,
    "norm_first": True,
    "batch_first": True,
    "tie_input_output_embeddings": True,
}


class SiglipVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, 196, 256)
        )
        self.output_norm = nn.LayerNorm(256)
        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

    def forward(self, patch_tokens):
        assert patch_tokens.shape[1:] == (196, 768)
        memory = (
            self.projection(patch_tokens)
            + self.position_embedding
        )
        return self.output_norm(memory)


class SharedCaptionTransformerDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(
            config["maximum_decoder_steps"]
        )
        self.token_embedding = nn.Embedding(
            self.vocabulary_size,
            self.d_model,
            padding_idx=self.pad_id,
        )
        self.position_embedding = nn.Embedding(
            self.maximum_decoder_steps,
            self.d_model,
        )
        self.embedding_dropout = nn.Dropout(config["dropout"])
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            dropout=config["dropout"],
            activation=config["activation"],
            batch_first=True,
            norm_first=True,
            bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer,
            num_layers=config["num_layers"],
            norm=None,
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(
            torch.zeros(self.vocabulary_size)
        )
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=self.d_model ** -0.5,
        )
        nn.init.normal_(
            self.position_embedding.weight,
            mean=0.0,
            std=0.02,
        )
        with torch.no_grad():
            self.token_embedding.weight[self.pad_id].zero_()
            self.output_bias.zero_()

    def forward(
        self,
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask=None,
    ):
        batch_size, sequence_length = decoder_input_ids.shape
        position_ids = torch.arange(
            sequence_length,
            device=decoder_input_ids.device,
        ).unsqueeze(0).expand(batch_size, -1)
        target = (
            self.token_embedding(decoder_input_ids)
            * math.sqrt(self.d_model)
            + self.position_embedding(position_ids)
        )
        target = self.embedding_dropout(target)
        causal_mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=decoder_input_ids.device,
            ),
            diagonal=1,
        )
        padding_mask = None
        if decoder_attention_mask is not None:
            padding_mask = ~decoder_attention_mask.bool()
        hidden = self.transformer_decoder(
            tgt=target,
            memory=visual_memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask,
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden,
            self.token_embedding.weight,
            self.output_bias,
        )


torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
full_siglip_model = SiglipModel.from_pretrained(
    SIGLIP_SOURCE_DIR,
    local_files_only=True,
).to(device)
SIGLIP_ENCODER = full_siglip_model.vision_model
del full_siglip_model
VISUAL_ADAPTER = SiglipVisualAdapter().to(device)
SHARED_DECODER = SharedCaptionTransformerDecoder(
    DECODER_CONFIG
).to(device)

try:
    from torch.torch_version import TorchVersion
    safe_context = torch.serialization.safe_globals([TorchVersion])
except Exception:
    from contextlib import nullcontext
    safe_context = nullcontext()
with safe_context:
    decoder_checkpoint = torch.load(
        SHARED_DECODER_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )
assert decoder_checkpoint["config"] == DECODER_CONFIG
load_result = SHARED_DECODER.load_state_dict(
    decoder_checkpoint["state_dict"],
    strict=True,
)
assert not load_result.missing_keys
assert not load_result.unexpected_keys

sample_records = VALIDATION_RECORDS[:2]
sample_images = []
for record in sample_records:
    with Image.open(
        VALIDATION_PATH_BY_ID[record["image_id"]]
    ) as image:
        sample_images.append(image.convert("RGB").copy())
pixel_values = SIGLIP_PROCESSOR(
    images=sample_images,
    return_tensors="pt",
)["pixel_values"].to(device)
sequences = [
    record["token_ids"]
    for record in sample_records
]
maximum_length = max(len(sequence) for sequence in sequences)
padded = [
    sequence + [0] * (maximum_length - len(sequence))
    for sequence in sequences
]
full_ids = torch.tensor(padded, dtype=torch.long, device=device)
decoder_input_ids = full_ids[:, :-1]
decoder_attention_mask = decoder_input_ids.ne(0)
labels = full_ids[:, 1:]

SIGLIP_ENCODER.eval()
VISUAL_ADAPTER.eval()
SHARED_DECODER.eval()
with torch.inference_mode():
    siglip_output = SIGLIP_ENCODER(
        pixel_values=pixel_values
    ).last_hidden_state
    patch_tokens = siglip_output
    visual_memory = VISUAL_ADAPTER(patch_tokens)
    logits = SHARED_DECODER(
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask,
    )
assert list(patch_tokens.shape) == [2, 196, 768]
assert list(visual_memory.shape) == [2, 196, 256]
assert logits.shape[:2] == decoder_input_ids.shape
assert logits.shape[-1] == 10080
assert torch.isfinite(logits).all()

initial_light_checkpoint_path = (
    CHECKPOINT_DIR
    / "teacher_c_adapter_decoder_initial_seed42.pt"
)
torch.save(
    {
        "schema_version": "phase3_teacher_c_light_initial_v1",
        "seed": SEED,
        "siglip_source_contract_sha256": sha256_file(
            CONFIG_DIR
            / "teacher_c_siglip_pretrained_source_contract.json"
        ),
        "decoder_config": DECODER_CONFIG,
        "adapter_state_dict": {
            key: value.detach().cpu()
            for key, value in VISUAL_ADAPTER.state_dict().items()
        },
        "decoder_state_dict": {
            key: value.detach().cpu()
            for key, value in SHARED_DECODER.state_dict().items()
        },
    },
    initial_light_checkpoint_path,
)

contract_path = CONFIG_DIR / "teacher_c_architecture_contract.json"
write_json(
    contract_path,
    {
        "contract_version": "phase3_teacher_c_architecture_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "encoder": "google/siglip-base-patch16-224 vision encoder",
        "encoder_output": "[B,196,768]",
        "cls_token_policy": "not_applicable",
        "patch_tokens": "[B,196,768]",
        "adapter": "Linear(768,256) + learned position + LayerNorm",
        "visual_memory": "[B,196,256]",
        "decoder": DECODER_CONFIG,
        "shared_decoder_strict_loaded": True,
        "initial_light_checkpoint": {
            "path": str(initial_light_checkpoint_path),
            "sha256": sha256_file(initial_light_checkpoint_path),
        },
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

audit_path = AUDIT_DIR / "phase3i_cell07_architecture.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell07_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "patch_token_shape": list(patch_tokens.shape),
        "visual_memory_shape": list(visual_memory.shape),
        "logits_shape": list(logits.shape),
        "decoder_strict_load": True,
        "forward_finite": True,
        "network_download": False,
        "optimizer_created": False,
        "optimizer_step_performed": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3I CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Encoder                   : SigLIP ViT-B/16")
print("SigLIP visual tokens         :", list(patch_tokens.shape))
print("Visual memory             :", list(visual_memory.shape))
print("Decoder strict load       : PASSED")
print("Logits                    :", list(logits.shape))
print("Forward finite            : True")
print("Network download          : False")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Architecture contract     :", contract_path)
print("Progress                  : 7 / 11 cells passed")
print("Next                      : forward, loss and backward smoke test")


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

PHASE 3I CELL 7 STATUS
Status                    : PASSED
Encoder                   : SigLIP ViT-B/16
SigLIP visual tokens         : [2, 196, 768]
Visual memory             : [2, 196, 256]
Decoder strict load       : PASSED
Logits                    : [2, 13, 10080]
Forward finite            : True
Network download          : False
Optimizer step performed  : False
Test Set status           : SEALED
Architecture contract     : /kaggle/working//phase3_teacher_c_siglip_preflight/configs/teacher_c_architecture_contract.json
Progress                  : 7 / 11 cells passed
Next                      : forward, loss and backward smoke test


## Cell 8 — Forward, CE và backward smoke test

**Công việc:** đóng băng SigLIP encoder; kiểm gradient adapter và decoder.

**Đầu ra dự kiến:** CE hữu hạn, gradient hữu hạn/không thiếu, encoder gradient
bằng 0 như thiết kế.


In [8]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "SIGLIP_ENCODER",
    "VISUAL_ADAPTER",
    "SHARED_DECODER",
    "pixel_values",
    "decoder_input_ids",
    "decoder_attention_mask",
    "labels",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–7: {missing}"

for parameter in SIGLIP_ENCODER.parameters():
    parameter.requires_grad = False
SIGLIP_ENCODER.eval()
VISUAL_ADAPTER.train()
SHARED_DECODER.train()
VISUAL_ADAPTER.zero_grad(set_to_none=True)
SHARED_DECODER.zero_grad(set_to_none=True)

with torch.no_grad():
    patch_tokens = SIGLIP_ENCODER(
        pixel_values=pixel_values
    ).last_hidden_state
visual_memory = VISUAL_ADAPTER(patch_tokens)
logits = SHARED_DECODER(
    decoder_input_ids,
    visual_memory,
    decoder_attention_mask,
)
loss = F.cross_entropy(
    logits.reshape(-1, 10080),
    labels.reshape(-1),
    ignore_index=0,
)
assert torch.isfinite(loss)
loss.backward()


def gradient_audit(model):
    missing = []
    nonfinite = []
    squared_sum = 0.0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if parameter.grad is None:
            missing.append(name)
            continue
        if not torch.isfinite(parameter.grad).all():
            nonfinite.append(name)
            continue
        squared_sum += (
            parameter.grad.detach().float().pow(2).sum().item()
        )
    return {
        "missing": missing,
        "nonfinite": nonfinite,
        "l2_norm": math.sqrt(squared_sum),
    }


adapter_gradients = gradient_audit(VISUAL_ADAPTER)
decoder_gradients = gradient_audit(SHARED_DECODER)
assert not adapter_gradients["missing"]
assert not adapter_gradients["nonfinite"]
assert adapter_gradients["l2_norm"] > 0
assert not decoder_gradients["missing"]
assert not decoder_gradients["nonfinite"]
assert decoder_gradients["l2_norm"] > 0
assert all(
    parameter.grad is None
    for parameter in SIGLIP_ENCODER.parameters()
)

audit_path = AUDIT_DIR / "phase3i_cell08_backward_smoke.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell08_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "mode": "frozen_clip_encoder",
        "validation_smoke_batch": int(pixel_values.shape[0]),
        "cross_entropy": float(loss.item()),
        "adapter_gradients": adapter_gradients,
        "decoder_gradients": decoder_gradients,
        "siglip_encoder_gradients": 0,
        "optimizer_created": False,
        "optimizer_step_performed": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3I CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Mode                      : frozen SigLIP encoder")
print("Cross-entropy             :", f"{loss.item():.8f}")
print("Adapter gradient L2       :", f"{adapter_gradients['l2_norm']:.8f}")
print("Decoder gradient L2       :", f"{decoder_gradients['l2_norm']:.8f}")
print("Adapter gradients         : PASSED")
print("Decoder gradients         : PASSED")
print("Encoder gradients         : 0 EXPECTED")
print("Optimizer created         : False")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Progress                  : 8 / 11 cells passed")
print("Next                      : deterministic inference and resource audit")


PHASE 3I CELL 8 STATUS
Status                    : PASSED
Mode                      : frozen SigLIP encoder
Cross-entropy             : 10.75101376
Adapter gradient L2       : 8.28540311
Decoder gradient L2       : 8.55107894
Adapter gradients         : PASSED
Decoder gradients         : PASSED
Encoder gradients         : 0 EXPECTED
Optimizer created         : False
Optimizer step performed  : False
Test Set status           : SEALED
Progress                  : 8 / 11 cells passed
Next                      : deterministic inference and resource audit


## Cell 9 — Determinism và resource profile

**Công việc:** chạy inference hai lần và đo tham số, peak GPU memory.

**Đầu ra dự kiến:** feature/memory/logit repeat error đều bằng 0.


In [9]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "SIGLIP_ENCODER",
    "VISUAL_ADAPTER",
    "SHARED_DECODER",
    "pixel_values",
    "decoder_input_ids",
    "decoder_attention_mask",
    "AUDIT_DIR",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–8: {missing}"

SIGLIP_ENCODER.eval()
VISUAL_ADAPTER.eval()
SHARED_DECODER.eval()
torch.cuda.reset_peak_memory_stats()


def deterministic_forward():
    with torch.inference_mode():
        patches = SIGLIP_ENCODER(
            pixel_values=pixel_values
        ).last_hidden_state
        memory = VISUAL_ADAPTER(patches)
        output_logits = SHARED_DECODER(
            decoder_input_ids,
            memory,
            decoder_attention_mask,
        )
    return patches, memory, output_logits


patches_1, memory_1, logits_1 = deterministic_forward()
patches_2, memory_2, logits_2 = deterministic_forward()
patch_error = (patches_1 - patches_2).abs().max().item()
memory_error = (memory_1 - memory_2).abs().max().item()
logits_error = (logits_1 - logits_2).abs().max().item()
assert patch_error == 0.0
assert memory_error == 0.0
assert logits_error == 0.0
assert torch.isfinite(logits_1).all()

encoder_parameters = sum(
    parameter.numel()
    for parameter in SIGLIP_ENCODER.parameters()
)
adapter_parameters = sum(
    parameter.numel()
    for parameter in VISUAL_ADAPTER.parameters()
)
decoder_parameters = sum(
    parameter.numel()
    for parameter in SHARED_DECODER.parameters()
)
total_parameters = (
    encoder_parameters
    + adapter_parameters
    + decoder_parameters
)
peak_gpu_memory_mib = (
    torch.cuda.max_memory_allocated() / (1024 ** 2)
)

audit_path = AUDIT_DIR / "phase3i_cell09_determinism_resources.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell09_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "repeat_errors": {
            "patch_tokens": patch_error,
            "visual_memory": memory_error,
            "logits": logits_error,
        },
        "parameters": {
            "encoder": encoder_parameters,
            "adapter": adapter_parameters,
            "decoder": decoder_parameters,
            "total": total_parameters,
        },
        "peak_gpu_memory_mib": peak_gpu_memory_mib,
        "optimizer_step_performed": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3I CELL 9 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Patch repeat error        :", patch_error)
print("Memory repeat error       :", memory_error)
print("Logits repeat error       :", logits_error)
print("Encoder parameters        :", f"{encoder_parameters:,}")
print("Adapter parameters        :", f"{adapter_parameters:,}")
print("Decoder parameters        :", f"{decoder_parameters:,}")
print("Total parameters          :", f"{total_parameters:,}")
print("Peak GPU memory           :", f"{peak_gpu_memory_mib:.2f} MiB")
print("Output tensors finite     : True")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Progress                  : 9 / 11 cells passed")
print("Next                      : lock Teacher C training contract")


PHASE 3I CELL 9 STATUS
Status                    : PASSED
Patch repeat error        : 0.0
Memory repeat error       : 0.0
Logits repeat error       : 0.0
Encoder parameters        : 92,884,224
Adapter parameters        : 247,552
Decoder parameters        : 5,759,328
Total parameters          : 98,891,104
Peak GPU memory           : 452.74 MiB
Output tensors finite     : True
Optimizer step performed  : False
Test Set status           : SEALED
Progress                  : 9 / 11 cells passed
Next                      : lock Teacher C training contract


## Cell 10 — Khóa contract cho giai đoạn readiness

**Công việc:** khóa dữ liệu, kiến trúc, quality gate và cấu hình dự kiến cho
Teacher C. Optimizer vẫn chưa được tạo.

**Đầu ra dự kiến:** matched Student là S3; selection chỉ dùng Validation.


In [10]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "ENTRY_CONTRACT",
    "CONFIG_DIR",
    "AUDIT_DIR",
    "write_json",
    "sha256_file",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–9: {missing}"

matched_seed42_cider = float(
    ENTRY_CONTRACT["teacher_families"]["C"][
        "validation_entry_targets"
    ]["matched_seed42_cider_d"]
)
quality_threshold = matched_seed42_cider + 0.001

TRAINING_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_c_training_validation_contract.json"
)
training_contract = {
    "contract_version": "phase3_teacher_c_training_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "teacher_id": "A",
    "teacher_name": "SigLIP ViT-B/16 + shared caption decoder",
    "dataset": {
        "name": "MS COCO 2014 only",
        "split_protocol": "Karpathy",
        "training_images_per_epoch": 113287,
        "captions_per_image_per_epoch": 1,
        "validation_images": 5000,
        "validation_references": 25010,
        "test_set_status": "sealed",
    },
    "architecture": {
        "encoder": "google/siglip-base-patch16-224",
        "adapter": "768_to_256_position_layernorm_196_tokens",
        "decoder": "phase2_shared_transformer_decoder",
        "visual_memory": "[B,196,256]",
    },
    "stages": [
        {
            "name": "frozen_encoder",
            "epochs": [1, 5],
            "trainable": ["visual_adapter", "decoder"],
        },
        {
            "name": "partial_unfreeze",
            "epochs": [6, 20],
            "trainable": [
                "siglip_last_2_transformer_blocks",
                "siglip_post_layernorm",
                "visual_adapter",
                "decoder",
            ],
        },
    ],
    "optimization": {
        "optimizer": "AdamW",
        "physical_batch_size": 8,
        "effective_batch_size": 64,
        "gradient_accumulation": 8,
        "learning_rates": {
            "encoder_unfrozen": 1e-5,
            "adapter": 1e-4,
            "decoder": 3e-4,
        },
        "weight_decay": 1e-4,
        "gradient_clip_norm": 1.0,
        "scheduler": "warmup_5_percent_plus_cosine",
        "minimum_lr_factor": 0.05,
        "mixed_precision": "FP16_GradScaler",
        "maximum_epochs": 20,
    },
    "validation": {
        "teacher_forced_cross_entropy": True,
        "generation": {
            "beam_size": 3,
            "length_penalty": 0.7,
            "maximum_decoder_steps": 31,
            "minimum_lexical_tokens": 2,
            "forbidden_token_ids": [0, 1, 3],
        },
        "metrics": [
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
            "CIDEr-D",
        ],
        "primary_metric": "CIDEr-D",
        "tie_break": [
            "BLEU-4 higher",
            "CE lower",
            "earlier epoch",
        ],
        "selection_split": "validation",
    },
    "quality_gate": {
        "matched_student_id": "S3",
        "matched_seed42_cider_d": matched_seed42_cider,
        "minimum_absolute_improvement": 0.001,
        "teacher_minimum_validation_cider_d": quality_threshold,
        "failure_policy": (
            "Retain only as documented auxiliary teacher hypothesis"
        ),
    },
    "reproducibility": {
        "development_seed": 42,
        "checkpoint_boundary": "optimizer_step",
        "atomic_checkpoint": True,
        "mid_epoch_resume": True,
        "pilot_before_full_training": True,
    },
    "data_policy": {
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "training_started": False,
    },
}
write_json(TRAINING_CONTRACT_PATH, training_contract)

audit_path = AUDIT_DIR / "phase3i_cell10_training_contract.json"
elapsed = time.perf_counter() - CELL_STARTED_AT
write_json(
    audit_path,
    {
        "schema_version": "phase3i_cell10_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "training_contract": str(TRAINING_CONTRACT_PATH),
        "training_contract_sha256": sha256_file(
            TRAINING_CONTRACT_PATH
        ),
        "teacher_minimum_validation_cider_d": quality_threshold,
        "optimizer_created": False,
        "optimizer_step_performed": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": elapsed,
    },
)

print("=" * 72)
print("PHASE 3I CELL 10 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Training contract         : LOCKED")
print("Stages                    : frozen 1–5, partial unfreeze 6–20")
print("Physical/effective batch  : 8 / 64")
print("Gradient accumulation     : 8")
print("LR encoder/adapter/decoder: 1e-5 / 1e-4 / 3e-4")
print("Selection                 : Validation CIDEr-D")
print("Matched S3 seed42 CIDEr-D :", f"{matched_seed42_cider:.9f}")
print("Teacher quality threshold :", f"{quality_threshold:.9f}")
print("Optimizer created         : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Contract                  :", TRAINING_CONTRACT_PATH)
print("Progress                  : 10 / 11 cells passed")
print("Next                      : package preflight bundle")


PHASE 3I CELL 10 STATUS
Status                    : PASSED
Training contract         : LOCKED
Stages                    : frozen 1–5, partial unfreeze 6–20
Physical/effective batch  : 8 / 64
Gradient accumulation     : 8
LR encoder/adapter/decoder: 1e-5 / 1e-4 / 3e-4
Selection                 : Validation CIDEr-D
Matched S3 seed42 CIDEr-D : 1.076798619
Teacher quality threshold : 1.077798619
Optimizer created         : False
Training started          : False
Test Set status           : SEALED
Contract                  : /kaggle/working//phase3_teacher_c_siglip_preflight/configs/teacher_c_training_validation_contract.json
Progress                  : 10 / 11 cells passed
Next                      : package preflight bundle


## Cell 11 — Manifest, checksum và marker chuyển Phase 03J

**Công việc:** lập manifest toàn bộ preflight, tự xác minh checksum và ghi
marker READY.

**Đầu ra dự kiến:** `PHASE3_TEACHER_C_SIGLIP_PREFLIGHT_READY.json`; tiến độ
`11/11`; bước kế tiếp là Phase 03J readiness, chưa phải full training.


In [11]:
CELL_STARTED_AT = time.perf_counter()
required = [
    "OUTPUT_ROOT",
    "MANIFEST_DIR",
    "TRAINING_CONTRACT_PATH",
    "CONFIG_DIR",
    "AUDIT_DIR",
    "write_json",
    "sha256_file",
]
missing = [name for name in required if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–10: {missing}"

required_audits = [
    AUDIT_DIR / f"phase3i_cell{index:02d}_{suffix}.json"
    for index, suffix in [
        (1, "environment_discovery"),
        (2, "phase3h_authorization"),
        (3, "upstream_artifacts"),
        (4, "siglip_source"),
        (5, "validation_index"),
        (6, "preprocessing"),
        (7, "architecture"),
        (8, "backward_smoke"),
        (9, "determinism_resources"),
        (10, "training_contract"),
    ]
]
for audit in required_audits:
    assert audit.is_file(), f"Thiếu audit: {audit}"
    assert read_json(audit)["status"] == "passed"

MANIFEST_CSV_PATH = (
    MANIFEST_DIR / "teacher_c_preflight_manifest.csv"
)
MANIFEST_JSON_PATH = (
    MANIFEST_DIR / "teacher_c_preflight_manifest.json"
)
SHA256SUMS_PATH = (
    MANIFEST_DIR / "teacher_c_preflight_SHA256SUMS.txt"
)
READY_PATH = (
    OUTPUT_ROOT / "PHASE3_TEACHER_C_SIGLIP_PREFLIGHT_READY.json"
)
excluded = {
    MANIFEST_CSV_PATH.resolve(),
    MANIFEST_JSON_PATH.resolve(),
    SHA256SUMS_PATH.resolve(),
    READY_PATH.resolve(),
}
artifact_rows = []
for artifact in sorted(
    OUTPUT_ROOT.rglob("*"),
    key=lambda value: value.as_posix(),
):
    if not artifact.is_file():
        continue
    if artifact.resolve() in excluded:
        continue
    if artifact.name.endswith(".tmp"):
        raise AssertionError(f"Temporary artifact: {artifact}")
    artifact_rows.append(
        {
            "relative_path": artifact.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "size_bytes": artifact.stat().st_size,
            "sha256": sha256_file(artifact),
        }
    )
assert artifact_rows

with open(
    MANIFEST_CSV_PATH,
    "w",
    encoding="utf-8",
    newline="",
) as target:
    writer = csv.DictWriter(
        target,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(artifact_rows)
write_json(
    MANIFEST_JSON_PATH,
    {
        "schema_version": "phase3_teacher_c_preflight_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(artifact_rows),
        "total_size_bytes": sum(
            row["size_bytes"] for row in artifact_rows
        ),
        "artifacts": artifact_rows,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
SHA256SUMS_PATH.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in artifact_rows
    ),
    encoding="utf-8",
)

with open(
    MANIFEST_CSV_PATH,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    verified_rows = list(csv.DictReader(source))
for row in verified_rows:
    artifact = resolve_inside(
        OUTPUT_ROOT,
        row["relative_path"],
    )
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

ready_payload = {
    "marker_version": "phase3_teacher_c_preflight_ready_v1",
    "created_at_utc": now_utc(),
    "status": "ready",
    "teacher_id": "A",
    "teacher": "SigLIP ViT-B/16 + shared caption decoder",
    "matched_student_id": "S3",
    "preflight_cells_passed": 11,
    "training_contract": {
        "relative_path": TRAINING_CONTRACT_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(TRAINING_CONTRACT_PATH),
    },
    "manifests": {
        "csv": {
            "relative_path": MANIFEST_CSV_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV_PATH),
        },
        "json": {
            "relative_path": MANIFEST_JSON_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_JSON_PATH),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHA256SUMS_PATH),
        },
    },
    "authorized_next_action": (
        "Create Phase 03J Teacher C training-readiness notebook. "
        "Run one-batch overfit, exact-resume and discarded throughput pilot ""before any full training."
    ),
    "training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
}
write_json(READY_PATH, ready_payload)

elapsed = time.perf_counter() - CELL_STARTED_AT
print("=" * 72)
print("PHASE 3I CELL 11 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher C preflight       : READY")
print("Audit cells               : 10 / 10 PASSED")
print("Manifest artifacts        :", len(artifact_rows))
print("Manifest verification     : ALL PASSED")
print("Pretrained source         : LOCKED")
print("Architecture              : LOCKED")
print("Preprocessing             : LOCKED")
print("Training contract         : LOCKED")
print("Optimizer step performed  : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("READY marker              :", READY_PATH)
print("READY SHA-256             :", sha256_file(READY_PATH))
print("Progress                  : 11 / 11 cells passed")
print("Elapsed                   :", round(elapsed, 2), "giây")
print("Next                      : Save Version, then build Phase 03J")


PHASE 3I CELL 11 STATUS
Status                    : PASSED
Teacher C preflight       : READY
Audit cells               : 10 / 10 PASSED
Manifest artifacts        : 15
Manifest verification     : ALL PASSED
Pretrained source         : LOCKED
Architecture              : LOCKED
Preprocessing             : LOCKED
Training contract         : LOCKED
Optimizer step performed  : False
Training started          : False
Test Set status           : SEALED
Test Set used             : False
Flickr data used          : False
READY marker              : /kaggle/working//phase3_teacher_c_siglip_preflight/PHASE3_TEACHER_C_SIGLIP_PREFLIGHT_READY.json
READY SHA-256             : b4dae6bc988720ba73a70b329aa8a364f7a24fd7db52463ddc444de53492ba2d
Progress                  : 11 / 11 cells passed
Elapsed                   : 0.14 giây
Next                      : Save Version, then build Phase 03J
